 # <center>Practice Notebook - Lesson 6a - `numpy` basics </center>


This notebook will work through some of the basics of the NumPy package, including

1. Making numpy containers from existing data
2. Creating numpy containers from scratch
3. Performing basic computations with numpy arrays -- without ever explicitly looping!!! Amazing!
4. Basics of plotting using Matplotlib / Pyplot.
5. Selecting data using slicing and indexing

---

References for the material in this notebook include 
* posted readings and media as listed on Canvas
* Chapter 4 of Python for Data Analysis, <em> numpy Basics: Arrays and Vectorized Computation</em>, 
* and the <a href="https://numpy.org/doc/stable/">official documentation for numpy</a>
* Section 9.1 of Python for Data Analysis, <em> Plotting and Visualization, A Brief matplotlib API Primer </em>

I have additional links throughout this notebook, too!

---


## 0. Import numpy to gain access

First, we'll import the numpy package. 

In [1]:
import numpy as np # canonical to import as `np`, shorter than `numpy`

There are many ways to create numpy arrays, including these options:
* convert from a list, tuple, or other Python collection to a numpy object
* construct directly in numpy
* read in data from a file (either directly, or by converting through a list or other collection)

---

## 1. Making arrays from lists and tuples

We'll start by showing how to create a numpy array from a regular old Python container, such as a list.

To create an ndarray from a sequence, we use the `np.array` function.  Suppose we have data in memory.  I'll use the high temperatures in Boulder again -- we last saw these in Lesson 4's notebook on tuples.

The daily high temperatures in degrees Fahrenheit for Boulder, CO in September of 2013 (<a href="https://psl.noaa.gov/boulder/data.daily.html">source</a>)

In [2]:
high_temps_as_tuple = (78, 90, 94, 93, 94, 95, 93, 91, 84, 65, 63, 67, 71, 74, 71, 72, 82, 83, 72, 71, 79, 80, 68, 75, 78, 67, 59, 67, 74, 81)

We want to be able to do numpy operations with the data, but it's in a tuple.  So we'll convert:

In [3]:
high_temps = np.array(high_temps_as_tuple)

Now we have the data in numpy.  Good! Let's inspect some properties of the container we just made:

In [4]:
print(high_temps.shape) # returns (30,) -- a tuple with a single number in it

(30,)


Notice that the resulting array is an array of dimension 1.  The shape is `(30,)`, indicating that there are 30 elements, and that it takes ONE index inside a `[]` to get to a number.  

The `.ndim` property is also the length of `.shape`:

In [5]:
assert high_temps.ndim == len(high_temps.shape) # a sanity check -- these must always be equal
print(f'high_temps has {high_temps.ndim} dimension(s)') # the number of indices it takes to get to a piece of data

high_temps has 1 dimension(s)


What if the data has more than one row?  Numpy can help us there, too!  Here's the rainfall data from that same time period, which we also saw in Lesson 4:

In [6]:
rainfall_as_tuple = (0.03, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 'T', 0.25, 1.02, 1.92, 9.08, 2.44, 0.01, 1.94, 0.49, 0.00, 0.02, 0.00, 
            0.00, 0.00, 0.04, 0.35, 0.00, 0.00, 0.00, 0.47, 0.10, 0.00, 0.00)

temp_rain_paired = list(zip(high_temps_as_tuple, rainfall_as_tuple))
# now we have a list-of-tuples

Convert the list-of-tuples to a numpy object by calling `np.array`:

In [7]:
temperature_rain = np.array(temp_rain_paired)
print(f'temperature_rain is of shape {temperature_rain.shape}, has {temperature_rain.ndim} dimension(s), with a total of {temperature_rain.size} elements')

temperature_rain is of shape (30, 2), has 2 dimension(s), with a total of 60 elements


Observe how there are TWO numbers in the `shape`.  That means it takes TWO indices inside `[]` to get to a single number.  

Also, observe that numpy made the temperatures the first column, and the rainfalls the second column.

Again, observe that `.ndim` is the length of the `.shape` property:

In [8]:
# a sanity check, providing evidence that what I wrote is correct.
assert temperature_rain.ndim == len(temperature_rain.shape)

We can get the data from September 1, 2013 by getting the first row of the container:

In [9]:
print(temperature_rain[0,:])

['78' '0.03']


We can get the temperature by specifying TWO indices (because `ndim` is 2):

In [10]:
for ii in range(temperature_rain.shape[0]): # .shape[0] is the number of rows
    print(f'the temperature on September {ii+1}, 2013 was {temperature_rain[ii,0]} degrees')
    if ii>4:
        break # stops being interesting after 5 or so

the temperature on September 1, 2013 was 78 degrees
the temperature on September 2, 2013 was 90 degrees
the temperature on September 3, 2013 was 94 degrees
the temperature on September 4, 2013 was 93 degrees
the temperature on September 5, 2013 was 94 degrees
the temperature on September 6, 2013 was 95 degrees


What if you wanted the rainfall to be the first ROW instead of the first COLUMN? 

Well, currently our data is in a column, and we want to make it a row.  This operation is called "transposing" -- and numpy makes it easy.

Just use the `.T` property!  Know that this does NOT modify the original array, but produces a new one that's the transpose of the old one.

In [11]:
# transpose to make rows into columns, and columns into rows
temperature_rain_as_rows = temperature_rain.T # subtle note: no () needed.  It's `T`, not `T()`.

In [12]:
# observe that now we have (2,30), but we used to have (30,2)
temperature_rain_as_rows.shape

(2, 30)

Now our temperatures are the first ROW, and we can get them with a slice:

In [13]:
temperatures = temperature_rain_as_rows[0,:] # : means everything in that dimension
print(temperatures)

['78' '90' '94' '93' '94' '95' '93' '91' '84' '65' '63' '67' '71' '74'
 '71' '72' '82' '83' '72' '71' '79' '80' '68' '75' '78' '67' '59' '67'
 '74' '81']


---

## 2. Making numpy arrays directly

We can also make numpy arrays directly, without making lists or tuples first.  This is good practice, especially when you want to just make a discretization of an interval, or regularly spaced data.

#### Arrays of integers using `arange`

The [`np.arange` function](https://realpython.com/how-to-use-numpy-arange/) is great for making numpy arrays of integers.  It's signature is 

```
numpy.arange([start, ]stop, [step, ], dtype=None)
```

1. The first input is `start`.  It's the lower bound of the range.
2. Next is `stop`.  This is the upper bound -- and it's half-open, just like `range`s for loops.
3. Then we have `step`.  This is the distance between elements in the generated container.  The default value is 1.
4. We can also specify the data type using `dtype`.  I rarely do this, and we need not concern ourselves with data types right now.

In [14]:
from_0_to_10 = np.arange(10)
print(from_0_to_10)
from_303_to_350 = np.arange(303,350)
print(from_303_to_350)

[0 1 2 3 4 5 6 7 8 9]
[303 304 305 306 307 308 309 310 311 312 313 314 315 316 317 318 319 320
 321 322 323 324 325 326 327 328 329 330 331 332 333 334 335 336 337 338
 339 340 341 342 343 344 345 346 347 348 349]


#### Discretize a line segment using `linspace`

Sometimes I would rather not manually specify the `step`, but indicate the line segment or interval, and the number of pieces I want to break it into.  This is what [`np.linspace`](https://www.sharpsightlabs.com/blog/numpy-linspace/) is for. Its signature is:

```
np.linspace(start, stop,num)
```

* `start` is the left bound of the interval
* `stop` is the right bout of the interval
* `num` is the number of points in the discretization INCLUDING THE LEFT AND RIGHT ENDPOINTS!!!

In contrast to `arange`, the `linspace` function DOES include both the left and right endpoints.

Some examples:

In [15]:
unit_interval_four_points = np.linspace(0,1,4)
print(unit_interval_four_points)

[0.         0.33333333 0.66666667 1.        ]


See how 0.5 is not a member?  That's because we have 4 POINTS, not 4 subintervals.  Fencepost problem!  

If you want 0.5, the midpoint, you have to use an odd number of points:

In [16]:
unit_interval_five_points = np.linspace(0,1,5)
print(unit_interval_five_points)

[0.   0.25 0.5  0.75 1.  ]


#### Zeros, ones, and empty's, oh my!

Often we want to make a container into which we'll later write data.  That's what the following functions are for:
* `np.zeros` -- make an array of zeros, given a shape tuple
* `np.ones` -- make an array of ones, given a shape tuple
* `np.zeros_like` -- an array of zeros, with same shape as existing numpy container
* `np.ones_like` -- an array of ones, with same shape as existing numpy container

In [17]:
zeros_four_by_two = np.zeros((4,2))
print(zeros_four_by_two)

[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]


We can also create ndarrays of all [---]s of the same shape and data type as another ndarray, where "[---]" can be "zeros", "ones", or any other value. For example:

In [18]:
print(np.zeros_like(zeros_four_by_two))
print()
print(np.ones_like(zeros_four_by_two))

[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]

[[1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]]


Preallocating using `np.zeros` etc, can be useful when you want to produce data in a loop, but don't want to grow a container as you loop.  Just preallocate, and write into it!

As an example, I'll allocate space for evaluating the trig functions, and then do it.

In [19]:
t = np.linspace(-np.pi, np.pi, 100) # discretize an interval (one period of a basic wave)

function_values = np.zeros((2,t.size)) # make space to write into

function_values[0,:] = np.sin(t) # use the first row for sine
function_values[1,:] = np.cos(t) # use the second row for cosine.  

Note also that we computed sine and cosine of the discritized interval, with 100 points, but didn't have to loop!  That's the power of numpy!!!!

---

## 3. Vectorization & statistical methods

I want to explore more of what we just did -- compute 100 values without looping at all.  The numpy library was invented to do this.  It's called "vectorization", and employs "operator overloading".  That is,

* `+` for two numpy arrays computes the element-wise sum
* `+` for a number and a numpy array adds the number to every element of the array
 

and similarly,
* `*` for two numpy arrays computes a matrix product
* `*` for a number and an array scales the contents of array by the number

The mathematical operators are interpreted by context -- they don't always do the same thing.  Their meaning of an operator depends on the operands.  This is the essence of operator overloading.

### Demonstrating numpy arithmetic using real data

The list `AZ_cities` gives four cities in Arizona. For each city, `temp_list` contains a list of the city's maximum daily temperatures in degrees Fahrenheit for each day of the month of October 2017, so that the corresponding array has 4 "rows" (dimension 0) and 31 "columns" (dimension 1) (Source: <a href="https://www.ncdc.noaa.gov/cdo-web/">NOAA Climate Data Online</a>).

In [20]:
AZ_cities = ['Flagstaff', 'Iron Springs', 'Mesa', 'Tucson']
temp_list = [[69, 68, 67, 71, 74, 71, 76, 73, 68, 64, 68, 68, 67, 71, 70, 73, 74, 72, 68, 66, 62, 71, 73, 69, 71, 70, 69, 65, 69, 68, 60],
            [76, 71, 76, 81, 81, 77, 83, 76, 67, 74, 77, 75, 75, 77, 80, 83, 82, 79, 78, 71, 68, 78, 85, 80, 80, 77, 76, 76, 77, 71, 68],
            [96, 95, 97, 98, 97, 95, 95, 94, 91, 94, 97, 94, 95, 92, 97, 99, 98, 94, 96, 91, 83, 91, 96, 100, 97, 94, 91, 90, 91, 87, 82],
            [96, 92, 95, 97, 97, 92, 92, 94, 84, 90, 96, 93, 93, 91, 93, 93, 95, 94, 93, 89, 81, 89, 94, 94, 93, 90, 87, 88, 89, 85, 78]
            ]

Below, we convert the list of temperatures to degrees Celsius using a nested list comprehension.  (We've rounded to make looking at the numbers easier -- in general, don't round)

In [21]:
celsius_temp_list=[[round((5/9)*(temp_list[i][j]-32),2) for j in range(31)] for i in range(4)]   

print(celsius_temp_list)

[[20.56, 20.0, 19.44, 21.67, 23.33, 21.67, 24.44, 22.78, 20.0, 17.78, 20.0, 20.0, 19.44, 21.67, 21.11, 22.78, 23.33, 22.22, 20.0, 18.89, 16.67, 21.67, 22.78, 20.56, 21.67, 21.11, 20.56, 18.33, 20.56, 20.0, 15.56], [24.44, 21.67, 24.44, 27.22, 27.22, 25.0, 28.33, 24.44, 19.44, 23.33, 25.0, 23.89, 23.89, 25.0, 26.67, 28.33, 27.78, 26.11, 25.56, 21.67, 20.0, 25.56, 29.44, 26.67, 26.67, 25.0, 24.44, 24.44, 25.0, 21.67, 20.0], [35.56, 35.0, 36.11, 36.67, 36.11, 35.0, 35.0, 34.44, 32.78, 34.44, 36.11, 34.44, 35.0, 33.33, 36.11, 37.22, 36.67, 34.44, 35.56, 32.78, 28.33, 32.78, 35.56, 37.78, 36.11, 34.44, 32.78, 32.22, 32.78, 30.56, 27.78], [35.56, 33.33, 35.0, 36.11, 36.11, 33.33, 33.33, 34.44, 28.89, 32.22, 35.56, 33.89, 33.89, 32.78, 33.89, 33.89, 35.0, 34.44, 33.89, 31.67, 27.22, 31.67, 34.44, 34.44, 33.89, 32.22, 30.56, 31.11, 31.67, 29.44, 25.56]]


Transforming loops into array expressions, where computations are performed on an entire array rather than looping through values, is called vectorization; this transformation can have a significant impact on runtime, especially in the case of numerical computations. 

Let's convert those lists to numpy arrays to gain access to numpy functionality:

In [22]:
AZ_cities = np.array(AZ_cities)
temperatures = np.array(temp_list)

print(temperatures.shape)

(4, 31)


We can see that we get a shape `(4,31)` two-dimensional array.  The rows are cities, the columns are days.

🧠 In the cell below use vectorization to convert the values in the ndarray `tempuratures` to degrees Celsius.   Use overloaded `+` and `*`, to implement the formula

\( c = \frac{5}{9} f - 32 \)

In [23]:
temperatures_celsius = # 🧠 your code here.  should be one line, no looping.  use `tempuratures`.

SyntaxError: invalid syntax (358815258.py, line 1)

Suppose we wish to find the average high temperature for the month of October across all four cities; we can use the `mean` function. 

In [24]:
temperatures.mean()

82.69354838709677

We can also take means across particular axes of our array: if we wish to find the mean high temperature in each city for the month of October, we specify `axis=1`, which takes means across the "columns" of the 2-dimensional array (i.e means within each row).

In [25]:
temperatures.mean(axis=1) # axis=1 means per row

array([69.19354839, 76.61290323, 93.77419355, 91.19354839])

🧠 What if we wished to find the maximum daily high temperature for October 2017, in each city? Try it out below.  Hints: `max`, `axis=0`.

In [ ]:
# your code here 🧠 

max_daily_temperature = # 🧠 
print(max_daily_temperature)

Other array statistical methods can be found in Table 4-5 of your text.

---

## 4. Pyplot basics

We will do more with plotting later, but note for now that `pyplot` allows us to create visualizations. We plot the temperature data for the four cities below.

We start by issuing the canonical line to import pyplot:

In [ ]:
import matplotlib.pyplot as plt

Next we make a plot.  

1. Start by making a figure and axes pair, by calling `plt.subplots()`.
2. Plot into the axes by calling `plt.plot()`, which makes line plots
3. Adjust axis labels
4. Make a legend
5. Save to disk by calling `fig.savefig()`

In [ ]:
fig, ax = plt.subplots() # make a figure and a set of axes

for ii in range(4):
    plt.plot(temperatures[ii], label=AZ_cities[ii]) # label for the legend later
    # pyplot will automatically set colors for you.  their choices are not accessible for people with colorblindness.
    
# label the axes
ax.set_xlabel('day')
ax.set_ylabel('temperature (deg f)') 
ax.legend() # make a legend

fig.savefig('AZ_temperatures.png') # I like png's or pdf's.

ℹ️ Note that you'll see no evidence that the figure was saved -- you have to open the file externally.  

ℹ️ In your `.py` code for assignments in which you plot, your code should save the fig, but not draw it.  You'll have to open the plot file separately to view it, but that's ok -- it means that you've automated the process of plotting, and could possibly do this in a loop and make an animation, for example.

---

## 5. Indexing and selecting data

#### Slicing

We'll first discuss indexing into numpy arrays.  Numpy arrays are superior to lists-of-lists for many reasons, one being the way we access data using `[]` (pronounced "at", because it describes the location of things).

Recall that we have a list-of-lists, `temp_list`.  It's a list of length 4, where each element is itself a list of length 31.  If we want the high temperature of each of the four cities on Halloween 2017, we have to engage using a clunky sequence of `[]` accesses:

In [ ]:
halloween_temps = [t[30] for t in temp_list]
print(halloween_temps)

Not my favorite.  Numpy is much better.  Check this out:

In [ ]:
halloween_temps = temperatures[:,30]
print(halloween_temps)

Much better!  I can use a single `[]`, and put multiple indices in.  In this case, I wanted all the rows (because rows are cities), and just the last element (31st, but 0-based, so `30`).  

Similarly, if I want just the data for the first city, I can use `[]` like this:

In [ ]:
temps_flagstaff = temperatures[0,:]
print(temps_flagstaff)

The `:` means "everything in this dimension".  And we fix the first index to get just that one row.

Ok, one more.  What if I wanted just the first ten days of data, for Flagstaff?  No problem!

In [ ]:
temps_flagstaff_first_ten = temperatures[0,0:10]

We can use `:` to "slice" out the particular data we want.  We'll get rectangular containers of data in response. 

#### Fancy indexing



Fancy indexing allows us to pull out and rearrange the values in one dimension of the ndarray. For example, suppose we wanted to order our temperature data so that it corresponded to the cities in reverse order instead:

In [ ]:
rev_alpha_temps = temperatures[[3, 2, 1, 0]]
print(rev_alpha_temps)

Note that fancy indexing will create a copy of the data into a new ndarray, so that we can change values without affecting the original ndarray. 

For example, suppose we replace Tucson's temperatures with the mean high temperature in Tucson during October; this will not change the values in `temperatures`.

In [ ]:
rev_alpha_temps[0] = rev_alpha_temps.mean(axis = 1)[0]
print(rev_alpha_temps)
print()
print(temperatures)

🧠 Observe what happens when we try to use fancy indexing on multiple dimensions of the data at a time. Try to determine where the returned values are coming from.

In [ ]:
rev_alpha_temps[[1, 3],[0, 30]]

🧠 How could you pull out rows with indices 1 and 3, and and columns with indices 0 and 30 to get a 2-dimensional ndarray rather than what was returned above? Try out your ideas below.

#### Boolean indexing

Suppose we wish to pull out just the high temperatures for cities with at least one "a" in their names. With only four cities in our dataset, we could do this manually; however, with larger datasets, it will be convenient to use Boolean indexing, which allows us to select rows with particular properties. First, we create an ndarray of Booleans that indicates the characteristic we want.

In [ ]:
print(AZ_cities)
print(np.char.count(AZ_cities, 'a') >= 1)

Then, we ask for all "rows" (dimension 0) that correspond to cities with at least one "a", and all "columns" (dimension 1).

In [ ]:
temperatures[np.char.count(AZ_cities, 'a') >= 1, :]

🧠 Try practicing Boolean indexing using a different characteristic below.

In [ ]:
# your code here


---

## When you're done

🧠 After completing the guided practice above, log onto Canvas to complete Quiz 6. You may wish to keep this notebook open while you work.